In [0]:
%sql
SELECT *
FROM workspace.rearc_lakeflow.gold_q1_population_stats;

In [0]:
%sql
SELECT
    series_id,
    series_label,
    year,
    quarterly_total
FROM workspace.rearc_lakeflow.gold_q2_productivity
ORDER BY series_id, year
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT series_id) AS distinct_series
FROM workspace.rearc_lakeflow.gold_q2_productivity;

In [0]:
%sql
SELECT
    year,
    annual_value,
    population
FROM workspace.rearc_lakeflow.gold_q3_bls_population
WHERE year BETWEEN 2013 AND 2018
ORDER BY year;

In [0]:
%sql
DESCRIBE TABLE workspace.rearc_lakeflow.silver_datausa_population;

In [0]:
%sql
SELECT
    COUNT(*) AS total_years,
    MIN(year) AS start_year,
    MAX(year) AS end_year,
    ROUND(AVG(population), 2) AS mean_population,
    ROUND(STDDEV_SAMP(population), 2) AS stddev_population
FROM workspace.rearc_lakeflow.silver_datausa_population
WHERE nation = 'United States'
  AND year BETWEEN 2013 AND 2018;

In [0]:
%sql
WITH quarterly_totals AS (
    SELECT
        TRIM(d.series_id) AS series_id,
        CAST(d.year AS INT) AS year,
        SUM(CAST(d.value AS DOUBLE)) AS quarterly_total
    FROM workspace.rearc_lakeflow.silver_bls_data d
    WHERE TRIM(d.period) IN ('Q01', 'Q02', 'Q03', 'Q04')
    GROUP BY
        TRIM(d.series_id),
        CAST(d.year AS INT)
),

ranked_results AS (
    SELECT
        q.series_id,
        s.series_label,
        q.year,
        q.quarterly_total,
        ROW_NUMBER() OVER (
            PARTITION BY q.series_id
            ORDER BY q.quarterly_total DESC, q.year ASC
        ) AS rn
    FROM quarterly_totals q
    INNER JOIN workspace.rearc_lakeflow.silver_bls_series s
        ON TRIM(s.series_id) = q.series_id
)

SELECT
    series_id,
    series_label,
    year,
    quarterly_total
FROM ranked_results
WHERE rn = 1
ORDER BY series_id;

In [0]:
%sql
SELECT
    CAST(b.year AS INT) AS year,
    CAST(b.value AS DOUBLE) AS annual_value,
    p.population
FROM workspace.rearc_lakeflow.silver_bls_data b
LEFT JOIN workspace.rearc_lakeflow.silver_datausa_population p
    ON CAST(b.year AS INT) = p.year
   AND p.nation = 'United States'
WHERE TRIM(b.series_id) = 'PRS30006032'
  AND TRIM(b.period) = 'Q01'
ORDER BY year;